# WHERE vs HAVING

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql-02-where-vs-having.ipynb)

Both keywords filter, but they act at different stages. `WHERE` filters **rows** before grouping.
`HAVING` filters **groups** after aggregation. Put the condition in the wrong one and the query either
fails or answers a different question.

Previous: [filtering and grouping](sql-01-filtering-and-grouping.ipynb). Next: [NULL handling](sql-03-null-handling.ipynb).

## Setup

The first code cells install PySpark if it is missing, detect Java, and start a local Spark session.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"pyspark": "pyspark>=3.5,<4", "setuptools": "setuptools>=70"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

In [ ]:
# Detect PySpark and Java. Spark needs a Java runtime that can actually start a JVM.
import importlib.util
import shutil
import subprocess

HAS_PYSPARK = importlib.util.find_spec("pyspark") is not None
HAS_JAVA = shutil.which("java") is not None

# `shutil.which` only proves the binary exists; probe that it can start a JVM.
JAVA_RUNS = False
if HAS_JAVA:
    try:
        probe = subprocess.run(["java", "-version"], capture_output=True, text=True, timeout=60)
        JAVA_RUNS = probe.returncode == 0
    except Exception:
        JAVA_RUNS = False

SPARK_AVAILABLE = HAS_PYSPARK and HAS_JAVA and JAVA_RUNS

print(f"PySpark installed : {HAS_PYSPARK}")
print(f"Java on PATH      : {HAS_JAVA}")
print(f"Java runnable     : {JAVA_RUNS}")
print(f"Spark available   : {SPARK_AVAILABLE}")
if not SPARK_AVAILABLE:
    print("SKIPPED: Spark is not available, so every example prints SKIPPED instead of running.")

In [ ]:
import os
import sys
import time
from datetime import date, datetime
from decimal import Decimal

# Spark timestamps and Python datetimes must agree on the clock, so pin the process to UTC.
os.environ["TZ"] = "UTC"
if hasattr(time, "tzset"):
    time.tzset()

if SPARK_AVAILABLE:
    from pyspark.sql import SparkSession

    os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
    spark = (
        SparkSession.builder.master("local[2]")
        .appName("sql-learning-path")
        .config("spark.sql.shuffle.partitions", "4")
        .config("spark.sql.session.timeZone", "UTC")
        .config("spark.ui.enabled", "false")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")
    print("Spark session ready:", spark.version)
else:
    spark = None


def create_view(name, ddl, rows):
    """Register rows as a temporary view. Skipped when Spark is unavailable."""
    if not SPARK_AVAILABLE:
        print(f"SKIPPED view {name}: needs a Spark session.")
        return
    spark.createDataFrame(rows, ddl).createOrReplaceTempView(name)
    print(f"view {name}: {spark.table(name).count()} rows")


def _plain(value):
    """Spark returns DECIMAL for literal arithmetic; compare those as floats."""
    return float(value) if isinstance(value, Decimal) else value


def check(query, expected, label):
    """Show a query's result, then assert it equals the expected rows. Skipped without Spark."""
    if not SPARK_AVAILABLE:
        print(f"SKIPPED {label}: needs a Spark session.")
        return
    frame = spark.sql(query)
    frame.show(truncate=False)
    rows = [tuple(_plain(value) for value in row) for row in frame.collect()]
    assert rows == expected, f"{label}: got {rows}, expected {expected}"
    print(f"PASS {label}")

## The example data

The same `customers` and `orders` tables as the previous notebook.

In [ ]:
create_view(
    "customers",
    "customer_id INT, name STRING, region STRING, signup_date DATE",
    [
        (1, "Acme Corp", "North", date(2022, 1, 5)),
        (2, "Beta LLC", "North", date(2022, 2, 10)),
        (3, "Cyan Inc", "South", date(2022, 3, 15)),
        (4, "Delta Co", "South", date(2022, 4, 20)),
        (5, "Echo Ltd", "West", date(2022, 5, 25)),
        (6, "Foxtrot GmbH", "West", date(2022, 6, 30)),
        (7, "Gamma SA", "East", date(2023, 1, 10)),
        (8, "Hotel BV", "East", date(2023, 2, 15)),
    ],
)

create_view(
    "orders",
    "order_id INT, customer_id INT, status STRING, amount INT, order_date DATE",
    [
        (1, 1, "completed", 100, date(2023, 1, 5)),
        (2, 1, "completed", 200, date(2023, 2, 10)),
        (3, 1, "completed", 150, date(2023, 3, 15)),
        (4, 2, "completed", 300, date(2023, 1, 20)),
        (5, 2, "cancelled", 250, date(2023, 2, 5)),
        (6, 2, "completed", 400, date(2023, 3, 20)),
        (7, 3, "completed", 120, date(2023, 1, 25)),
        (8, 3, "completed", 180, date(2023, 2, 28)),
        (9, 4, "pending", 220, date(2023, 2, 14)),
        (10, 4, "completed", 320, date(2023, 3, 30)),
        (11, 5, "completed", 500, date(2023, 1, 2)),
        (12, 5, "completed", 450, date(2023, 3, 5)),
        (13, 6, "completed", 275, date(2023, 2, 18)),
        (14, 6, "cancelled", 325, date(2023, 3, 25)),
    ],
)

## The rule

- `WHERE` sees one input row at a time. It cannot use `SUM(...)`, `COUNT(...)` or any other aggregate.
- `HAVING` sees one group at a time, so it can use aggregates. It can also use grouped columns.

A useful test: if the condition is about one order, it belongs in `WHERE`. If it is about a customer's
total or count, it belongs in `HAVING`.

## Example 1: totals per customer

"Customers whose completed revenue is above 500." The condition `status = 'completed'` is about each order,
so it goes in `WHERE`. The threshold is about each customer's total, so it goes in `HAVING`.

In [ ]:
check(
    """
    SELECT customer_id, SUM(amount) AS completed_revenue
    FROM orders
    WHERE status = 'completed'
    GROUP BY customer_id
    HAVING SUM(amount) > 500
    ORDER BY customer_id
    """,
    [(2, 700), (5, 950)],
    "completed revenue above 500",
)

## Example 2: the filter moves the answer

Filtering rows before grouping changes which orders are counted. Compare "customers with at least two
orders" with and without the completed-only filter. The second version counts fewer orders per customer,
so customers 4 and 6 drop out.

In [ ]:
check(
    """
    SELECT customer_id, COUNT(*) AS orders
    FROM orders
    GROUP BY customer_id
    HAVING COUNT(*) >= 2
    ORDER BY customer_id
    """,
    [(1, 3), (2, 3), (3, 2), (4, 2), (5, 2), (6, 2)],
    "at least two orders, any status",
)

In [ ]:
check(
    """
    SELECT customer_id, COUNT(*) AS completed_orders
    FROM orders
    WHERE status = 'completed'
    GROUP BY customer_id
    HAVING COUNT(*) >= 2
    ORDER BY customer_id
    """,
    [(1, 3), (2, 2), (3, 2), (5, 2)],
    "at least two completed orders",
)

## Example 3: `HAVING` without `GROUP BY`

Without a `GROUP BY`, the whole table is one group. `HAVING` then tests that single total. This is a
handy way to guard a pipeline: fail the query when the total is suspiciously small.

In [ ]:
check(
    """
    SELECT SUM(amount) AS total_amount
    FROM orders
    HAVING SUM(amount) > 1000
    """,
    [(3790,)],
    "whole-table total above 1000",
)

## Common mistakes

- **Aggregate in `WHERE`.** `WHERE SUM(amount) > 500` is rejected because the sum does not exist yet at that
  stage. Move it to `HAVING`.
- **Row condition in `HAVING`.** `HAVING status = 'completed'` either fails (status is not grouped) or
  keeps whole groups that happen to contain the value. Put row conditions in `WHERE`.
- **Aggregating before filtering rows.** Excluded rows are already counted once a group is aggregated, so a
  group-level filter cannot remove them from the total. Keep row filters in `WHERE` so every aggregate sees the
  same population.

## Takeaways

- Row-level condition: `WHERE`. Group-level condition: `HAVING`.
- Where you filter changes the numbers, not just the speed. Decide the population first, then aggregate.